In [14]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split

con = sqlite3.connect('../database/bank_churn.db')
df = pd.read_sql("SELECT * FROM bank_customers", con)
con.close()

cols_to_drop = [
    'rownumber', 'customerid', 'surname', 'first_name',
    'address', 'contact_information', 'date_of_birth',
    'churn_reason', 'churn_date', 'occupation'
]

df_clean = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Size before cleaning: {df.shape[1]}")
print(f"Size after cleaning: {df_clean.shape[1]}")

df_clean.head()

Size before cleaning: 25
Size after cleaning: 15


,gender,marital_status,number_of_dependents,income,education_level,customer_tenure,customer_segment,preferred_communication_channel,credit_score,credit_history_length,outstanding_loans,churn_flag,balance,numofproducts,numcomplaints
0,Male,Divorced,3,77710.14,High School,30,Retail,Phone,397,24,41959.74,0,211359.05,1,0
1,Female,Married,1,58209.87,High School,27,SME,Email,665,10,8916.67,0,30624.76,4,1
2,Female,Single,1,9794.01,High School,14,Retail,Email,715,21,43270.54,0,111956.61,2,6
3,Female,Divorced,5,15088.98,High School,23,Corporate,Phone,747,17,17887.65,0,201187.61,1,0
4,Female,Divorced,2,60726.56,Master's,22,Corporate,Email,549,25,32686.84,0,60391.24,5,6


In [15]:
X = df_clean.drop(columns=['churn_flag'])
y = df_clean['churn_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (92512, 14)
Test shape: (23128, 14)


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False
).set_output(transform="pandas")

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Data was successfully transformed!")
X_train_prep.head()

Numerical columns: ['number_of_dependents', 'income', 'customer_tenure', 'credit_score', 'credit_history_length', 'outstanding_loans', 'balance', 'numofproducts', 'numcomplaints']
Categorical columns: ['gender', 'marital_status', 'education_level', 'customer_segment', 'preferred_communication_channel']
Data was successfully transformed!


,number_of_dependents,income,customer_tenure,credit_score,credit_history_length,outstanding_loans,balance,numofproducts,numcomplaints,gender_Female,...,marital_status_Single,education_level_Bachelor's,education_level_Diploma,education_level_High School,education_level_Master's,customer_segment_Corporate,customer_segment_Retail,customer_segment_SME,preferred_communication_channel_Email,preferred_communication_channel_Phone
16758,0.875285,0.894357,-0.747594,1.099414,0.053986,0.768380,0.250714,-0.708177,-0.947944,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
26845,0.875285,0.843119,-1.556051,-1.339323,-0.061629,1.124395,1.314985,-0.708177,-1.264690,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
34033,-1.467638,1.590272,-0.863087,-1.194759,-1.333399,1.118898,0.607346,0.708912,0.952532,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
101171,-0.881907,1.682532,1.677780,-1.169617,1.094525,-0.093809,-1.005918,-1.416721,0.635786,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
103733,1.461015,-0.403573,-0.516606,-1.490173,0.053986,0.179521,-1.300007,0.000368,-0.947944,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [17]:
object_cols = X_train.select_dtypes(include=['object']).columns
print(X_train[object_cols].nunique().sort_values(ascending=False))

education_level                    4
customer_segment                   3
marital_status                     3
gender                             2
preferred_communication_channel    2
dtype: int64
